# DTX-AI — T5 Anomaly Explanation Fine-Tuning

Fine-tunes a T5-family sequence-to-sequence model to translate LSTM-AE inference
output into a two-paragraph natural-language explanation.

- **Input** (`input_text`): the model signals — predicted class, confidence,
  reconstruction MSE, the six raw logits, and the raw sensor features. The
  string is produced by `build_t5_samples.py`, the single source of truth for
  the input format, and already carries the `explain anomaly:` task prefix.
- **Target** (`target_text`): the reference explanation, produced by
  `build_t5_targets.py`.


## 1. Environment Setup

In [39]:
!pip install -q transformers datasets sentencepiece accelerate evaluate rouge_score

## 2. Imports & Device Detection

In [40]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA L4


## 3. Configuration

In [41]:
# --- Model ---
MODEL_NAME      = "t5-small"

# --- Data ---
CSV_PATH        = "t5_dataset.csv"    # columns: input_text, target_text, predicted_class
TEST_SIZE       = 0.15
SEED            = 42

# --- Tokenization lengths (sized to the dataset, with headroom) ---
MAX_INPUT_LEN   = 320
MAX_TARGET_LEN  = 256

# --- Task prefix ---
# input_text already begins with "explain anomaly: ", so this stays empty to
# avoid prefixing twice. Set it only if a model needs an extra instruction.
PREFIX          = ""

# --- Training ---
EPOCHS          = 30
BATCH_SIZE      = 8
LEARNING_RATE   = 3e-4
WARMUP_STEPS    = 100
WEIGHT_DECAY    = 0.01
PATIENCE        = 5                    # early-stopping patience (epochs)

# --- Output paths ---
OUTPUT_DIR      = "./dtx_ai_t5_output"
FINAL_DIR       = "./dtx_ai_t5_final"

With early stopping enabled, `EPOCHS` is an upper bound rather than a fixed
count: training stops once validation ROUGE-L has not improved for `PATIENCE`
epochs and the best checkpoint is restored.

## 4. Load Dataset

In [42]:
df = pd.read_csv(CSV_PATH)
required = {"input_text", "target_text"}
assert required.issubset(df.columns), f"missing columns: {required - set(df.columns)}"
df = df.dropna(subset=["input_text", "target_text"]).reset_index(drop=True)

print(f"Rows: {len(df)}")
if "predicted_class" in df.columns:
    print("Class distribution:")
    print(df["predicted_class"].value_counts().to_string())

Rows: 1000
Class distribution:
predicted_class
overload    500
nominal     500


## 5. Load Model & Tokenizer

In [43]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f"Loaded {MODEL_NAME}: {model.num_parameters():,} parameters")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loaded t5-small: 60,506,624 parameters


## 6. Data Preparation & Tokenization

The train/validation split is stratified on `predicted_class` (when present) so every class is represented in validation.

In [44]:
stratify_col = df["predicted_class"] if "predicted_class" in df.columns else None
train_df, val_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=SEED, stratify=stratify_col,
)
print(f"Train: {len(train_df)} | Validation: {len(val_df)}")

train_ds = Dataset.from_pandas(train_df[["input_text", "target_text"]].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[["input_text", "target_text"]].reset_index(drop=True))


def preprocess(batch):
    """Tokenize inputs and targets; mask padding in labels so it is ignored in loss."""
    inputs = [PREFIX + text for text in batch["input_text"]]
    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LEN, padding="max_length", truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN, padding="max_length", truncation=True,
    )
    model_inputs["labels"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
        for label in labels["input_ids"]
    ]
    return model_inputs


train_tok = train_ds.map(preprocess, batched=True, remove_columns=["input_text", "target_text"])
val_tok   = val_ds.map(preprocess, batched=True, remove_columns=["input_text", "target_text"])
print("Tokenization complete")

Train: 850 | Validation: 150


Map:   0%|          | 0/850 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenization complete


## 7. Evaluation Metrics

Reports ROUGE-1, ROUGE-2 and ROUGE-L between generated and reference explanations.

In [45]:
rouge = evaluate.load("rouge")


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True,
    )
    return {k: round(v * 100, 2) for k, v in result.items()}

## 8. Training Arguments

In [46]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_dir="./logs",
    logging_steps=20,
    save_total_limit=2,
    fp16=False,
    seed=SEED,
    report_to="none",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 9. Training

Early stopping restores the best checkpoint by validation ROUGE-L.

In [47]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print("Starting training...")
train_result = trainer.train()
print("Training complete")
print(train_result.metrics)

Starting training...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.605511,0.138077,63.450000,41.400000,53.180000,53.150000
2,0.141320,0.096039,66.690000,43.890000,56.530000,56.530000
3,0.115634,0.093189,64.390000,40.380000,53.980000,53.930000
4,0.108828,0.088215,65.010000,43.800000,55.420000,55.310000
5,0.100724,0.085023,64.500000,41.770000,54.380000,54.360000
6,0.099357,0.081909,62.580000,40.690000,53.060000,53.060000
7,0.095810,0.081904,63.510000,41.140000,53.840000,53.840000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Training complete
{'train_runtime': 457.7116, 'train_samples_per_second': 55.712, 'train_steps_per_second': 7.013, 'total_flos': 503302324224000.0, 'train_loss': 0.5460851043661701, 'epoch': 7.0}


## 10. Save Model & Final Evaluation

In [48]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("Saved to", FINAL_DIR)

metrics = trainer.evaluate()
print("Final validation metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./dtx_ai_t5_final


Final validation metrics:
  eval_loss: 0.09603861719369888
  eval_rouge1: 66.69
  eval_rouge2: 43.89
  eval_rougeL: 56.53
  eval_rougeLsum: 56.53
  eval_runtime: 46.0652
  eval_samples_per_second: 3.256
  eval_steps_per_second: 0.412
  epoch: 7.0


## 11. Inference Function & Sample Predictions

Generate explanations for a few validation rows and compare against the references.

In [49]:
model.to(device)
model.eval()


def generate_explanation(input_text, num_beams=4, max_length=MAX_TARGET_LEN):
    enc = tokenizer(
        PREFIX + input_text, return_tensors="pt",
        max_length=MAX_INPUT_LEN, truncation=True,
    ).to(device)
    with torch.no_grad():
        out = model.generate(**enc, max_length=max_length, num_beams=num_beams)
    return tokenizer.decode(out[0], skip_special_tokens=True)


for i in range(3):
    row = val_df.iloc[i]
    print("=" * 80)
    print("INPUT :", row["input_text"][:180], "...")
    print("-" * 80)
    print("PRED  :", generate_explanation(row["input_text"]))
    print("-" * 80)
    print("TARGET:", row["target_text"])
print("=" * 80)

INPUT : explain anomaly: predicted_class: nominal | confidence: 0.4232 | reconstruction_mse: 0.0321 | logits: nominal=0.97, bearing_wear=-0.09, overheat=-1.55, overload=0.27, pressure_faul ...
--------------------------------------------------------------------------------
PRED  : Running this reading through the autoencoder yields a prediction of nominal at 0.4232 confidence, an output that may — with low confidence — point to normal operating behaviour. The margin between nominal and overload in the logit vector is small, indicating a less clear-cut decision. With a reconstruction error of just 0.031, the reading is well represented by the model's learned manifold. Recorded values for this snapshot include vibration magnitude at 9.81 and line pseudo-pressure at -9237 Pa. This is a per-snapshot judgement, so it describes the current reading rather than behaviour over time. No operator action is required; the asset remains within its normal envelope.
-----------------------------------

## 12. Optional — Package for Download / Drive

In [38]:
# Zip the final model for download or upload to Drive.
# !zip -r dtx_ai_t5_final.zip {FINAL_DIR}

# from google.colab import drive
# drive.mount("/content/drive")
# !cp dtx_ai_t5_final.zip /content/drive/MyDrive/

  adding: dtx_ai_t5_final/ (stored 0%)
  adding: dtx_ai_t5_final/tokenizer_config.json (deflated 85%)
  adding: dtx_ai_t5_final/model.safetensors (deflated 38%)
  adding: dtx_ai_t5_final/generation_config.json (deflated 59%)
  adding: dtx_ai_t5_final/training_args.bin (deflated 53%)
  adding: dtx_ai_t5_final/config.json (deflated 49%)
  adding: dtx_ai_t5_final/tokenizer.json (deflated 77%)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
